#**Collaborative Filtering**

In [ ]:
# Import PySpark components and auxiliary libraries for data manipulation and analysis
from google.colab import drive

import os
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count

from pyspark.sql import functions as F

from pyspark.sql.functions import collect_list, sort_array, udf, slice, posexplode, hash, explode, col, row_number, array, sum as spark_sum, sqrt, pow, when

from pyspark.sql.window import Window

from pyspark.sql.types import ArrayType, LongType

In [ ]:
# Mount Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Find Project Folder
def find_project_folder(folder_name="CineFusion", base_path="/content/drive/MyDrive"):
    for root, dirs, files in os.walk(base_path):
        if folder_name in dirs:
            return os.path.join(root, folder_name)
    return None

BASE_DIR = find_project_folder()

if BASE_DIR is None:
    raise Exception("Project folder not found. Make sure it's added to MyDrive.")

print("Project folder found at:", BASE_DIR)

Project folder found at: /content/drive/MyDrive/CS 483/CineFusion


In [ ]:
# Start Spark Session
spark = SparkSession.builder \
    .appName("CineFusion - CF") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .getOrCreate()

In [ ]:
# Import Rating Data from the 25M Dataset
ratings = spark.read.csv(
    f"{BASE_DIR}/ml-25m/ratings.csv",
    # f"{BASE_DIR}/ml-100k/ratings-small.csv",
    header=True,
    inferSchema=True
).select("userId", "movieId", "rating")

movies = spark.read.csv(
    f"{BASE_DIR}/ml-25m/movies.csv",
    # f"{BASE_DIR}/ml-100k/movies-small.csv",
    header=True,
    inferSchema=True
).select("movieId", "title")

# print("Ratings count:", ratings.count())
# print("Movies count:", movies.count())

In [ ]:
# Counting unique entries
# ratings_stats = ratings.agg(
#     F.min("userId").alias("min_userId"),
#     F.max("userId").alias("max_userId"),
#     F.min("movieId").alias("min_movieId"),
#     F.max("movieId").alias("max_movieId"),
#     F.countDistinct("userId").alias("unique_userId"),
#     F.countDistinct("movieId").alias("unique_movieId")
# ).collect()[0]

# movies_stats = movies.agg(
#     F.min("movieId").alias("min_movieId"),
#     F.max("movieId").alias("max_movieId"),
#     F.countDistinct("movieId").alias("unique_movieId")
# ).collect()[0]

# print("rating.csv")
# print(f"userId : {ratings_stats['min_userId']} -> {ratings_stats['max_userId']}")
# print(f"movieId : {ratings_stats['min_movieId']} -> {ratings_stats['max_movieId']}")
# print(f"unique userId : {ratings_stats['unique_userId']}")
# print(f"unique movieId : {ratings_stats['unique_movieId']}")

# print("\nmovies.csv")
# print(f"movieId : {movies_stats['min_movieId']} -> {movies_stats['max_movieId']}")
# print(f"unique movieId : {movies_stats['unique_movieId']}")

In [ ]:
# Group all users who rated a movie and sort them
movie_users = ratings.groupBy("movieId") \
    .agg(sort_array(collect_list("userId")).alias("users"))

# movie_users.show(5)

In [ ]:
# Count total number of unique users who rated a movie
all_users = movie_users.select(explode("users").alias("userId")).distinct()

all_users_list = [row.userId for row in all_users.collect()]

# print("Number of unique users:", len(all_users_list))

In [ ]:
# Generate 100 Permutations of all uniques movies arranged in a random order
k = 100
np.random.seed(42)
permutations = [np.random.permutation(all_users_list).tolist()
                for _ in range(k)]
permutations_broadcast = spark.sparkContext.broadcast(permutations)

In [ ]:
# Minhashing
def minhash_signature(users):
    perms = permutations_broadcast.value
    sig = []
    user_set = set(users)

    for perm in perms:
        for idx, user in enumerate(perm):
            if user in user_set:
                sig.append(idx)
                break
    return sig

minhash_udf = udf(minhash_signature, ArrayType(LongType()))

movie_minhash = movie_users.withColumn(
    "minhash", minhash_udf("users")
).select("movieId", "minhash")

# movie_minhash.show(10)

In [ ]:
movie_minhash.write.parquet("/content/checkpoints/movie_minhash")

In [ ]:
# Making 20 Bands
b = 20
r = 5
assert b * r == 100

In [ ]:
# Distributing minhash according to bands
df = movie_minhash

for i in range(b):
    df = df.withColumn(
        f"band_{i}",
        slice(col("minhash"), i * r + 1, r)
    )

# df.show()

In [ ]:
df = df.cache()
df.count()

In [ ]:
# Reformating in row format
band_cols = [f"band_{i}" for i in range(b)]

df_bands = df.select(
    "movieId",
    posexplode(array(*[col(c) for c in band_cols])).alias("band_id", "band_values")
)

# df_bands.show()

In [ ]:
df_bands = df_bands.cache()
df_bands.count()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Hashing each band_values for all movieID and bandID
df_hashed = df_bands.withColumn(
    "bucket",
    hash(col("band_values"))
)

# df_hashed.show()

In [ ]:
df_hashed = df_hashed.cache()
df_hashed.count()

In [ ]:
# df_hashed.coalesce(1).write \
#     .mode("append") \
#     .parquet(f"{BASE_DIR}/ml-100k-output/df_hashed")

# print(f"Saved {df_hashed.count()} candidate pairs to {BASE_DIR}/ml-100k-output/df_hashed")

In [ ]:
# Putting similar movies in the bucket
# If two movies have identical values in ANY one band,
# they are considered a candidate pair
df_buckets = df_hashed.groupBy("band_id", "bucket") \
    .agg(collect_list("movieId").alias("movies")) \
    .filter("size(movies) > 1")

# df_buckets.show()

In [ ]:
df_buckets = df_buckets.cache()
df_buckets.count()

In [ ]:
# Get All Candidate pairs
df_pairs = df_buckets.select(
    "band_id",
    explode("movies").alias("movie1"),
    col("movies").alias("all_movies_in_bucket")
).select(
    "band_id",
    "movie1",
    explode("all_movies_in_bucket").alias("movie2")
).filter("movie1 < movie2")

# df_pairs.show()

In [ ]:
candidate_pairs = df_pairs.select("movie1", "movie2").distinct()

# candidate_pairs.show()

# print("Candidate pairs:", candidate_pairs.count())

In [ ]:
candidate_pairs = candidate_pairs.cache()
candidate_pairs.count()

In [ ]:
# candidate_pairs.coalesce(1).write \
#     .mode("append") \
#     .parquet(f"{BASE_DIR}/ml-100k-output/candidate_pairs_parquet_2")

# print(f"Saved {candidate_pairs.count()} candidate pairs to {BASE_DIR}/ml-100k-output/candidate_pairs_parquet_2")

In [ ]:
# Prepare ratings twice
# We create two versions of the ratings table:
# One for movie1 and one for movie2 so we can compare ratings
# given by the SAME user on BOTH movies

ratings1 = ratings.select(
    col("userId"),
    col("movieId").alias("movie1"),
    col("rating").alias("rating1")
)

ratings2 = ratings.select(
    col("userId"),
    col("movieId").alias("movie2"),
    col("rating").alias("rating2")
)


# Join candidate pairs with ratings
# For each (movie1, movie2), find users who rated BOTH movies

joined = candidate_pairs \
    .join(ratings1, "movie1") \
    .join(ratings2, ["userId", "movie2"])

# Now we have:
# (userId, movie1, movie2, rating1, rating2)

In [ ]:
# Compute mean rating per movie
# Pearson requires mean-centering per item (movie)

movie_means = ratings.groupBy("movieId").agg(
    avg("rating").alias("mean_rating")
)

# Split into two tables for movie1 and movie2
means1 = movie_means.select(
    col("movieId").alias("movie1"),
    col("mean_rating").alias("mean1")
)

means2 = movie_means.select(
    col("movieId").alias("movie2"),
    col("mean_rating").alias("mean2")
)

joined = joined.drop("mean1", "mean2")

# Join means into the dataset
joined = joined \
    .join(means1, "movie1") \
    .join(means2, "movie2")

# Mean-center the ratings
# Subtract average rating of each movie

centered = joined.withColumn(
    "r1_centered", col("rating1") - col("mean1")
).withColumn(
    "r2_centered", col("rating2") - col("mean2")
)

# Compute Pearson components
# numerator = sum(r1_centered * r2_centered)
# denominator = sqrt(sum(r1_centered^2)) * sqrt(sum(r2_centered^2))

similarities = centered.groupBy("movie1", "movie2").agg(
    spark_sum(col("r1_centered") * col("r2_centered")).alias("numerator"),
    sqrt(spark_sum(pow(col("r1_centered"), 2))).alias("denom1"),
    sqrt(spark_sum(pow(col("r2_centered"), 2))).alias("denom2")
)

# Final Pearson similarity
# Handle division-by-zero safely

similarities = similarities.withColumn(
    "similarity",
    when(
        (col("denom1") * col("denom2")) != 0,
        col("numerator") / (col("denom1") * col("denom2"))
    ).otherwise(0)
)

# Optional: cache because we'll reuse it
similarities.cache()

DataFrame[movie1: int, movie2: int, numerator: double, denom1: double, denom2: double, similarity: double]

In [ ]:
# Make similarity symmetric
# If movie1 ~ movie2, then movie2 ~ movie1

sim1 = similarities.select(
    col("movie1"),
    col("movie2"),
    col("similarity")
)

sim2 = similarities.select(
    col("movie2").alias("movie1"),
    col("movie1").alias("movie2"),
    col("similarity")
)

similarities_full = sim1.union(sim2)

# Keep top 100 neighbors per movie
# Rank similar movies for each movie

window = Window.partitionBy("movie1").orderBy(col("similarity").desc())

top100 = similarities_full.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 100)


#  Keep top 10 recommendations
top10 = top100.filter(col("rank") <= 10)


# Attach movie titles (optional but useful)
top10_with_titles = top10 \
    .join(movies.withColumnRenamed("movieId", "movie1"), "movie1") \
    .withColumnRenamed("title", "movie1_title") \
    .join(movies.withColumnRenamed("movieId", "movie2"), "movie2") \
    .withColumnRenamed("title", "movie2_title")


# Final output
top10_with_titles.select(
    "movie1",
    "movie1_title",
    "movie2",
    "movie2_title",
    "similarity",
    "rank"
).show(20, truncate=False)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
def get_top_10_similar(movie_id):

    top10 = similarities_full \
        .filter(col("movie1") == movie_id) \
        .orderBy(col("similarity").desc()) \
        .limit(10)

    result = top10 \
        .join(
            movies.withColumnRenamed("movieId", "movie2"),
            "movie2"
        ) \
        .select(
            "movie2",
            "title",
            "similarity"
        )

    return result

In [ ]:
movie_id = 19  # change this
get_top_10_similar(movie_id).show(truncate=False)